# clock/`__goal_oracle_refinement` — tightening the GoalScope faithfulness / causality oracles

The current faithfulness & causality rubrics (`scripts/evaluation/eval_goalscope.py`,
`FAITH_HARD_SYS` / `CAUSAL_HARD_SYS`) are **very lenient**: any relevant fragment → `YES`, and
looping text is explicitly kept as `YES` *"as long as the repeated content is relevant"*.

`resid_norm` read-outs are dominated by **degenerate short loops**
(`value >= value >= value >= …`, `len len len …`, `sum sum ( sum % sum == 0` on repeat) that the
v1 rubric scores `YES` even though the verbalization is unintelligible. This inflates the residual
YES-rate and muddies the head-vs-residual comparison.

**Fix (v2):** add a `HEAVY_LOOP` verdict, checked *before* `YES`, for read-outs dominated by a
repeating cycle of ≲5 words — regardless of whether the looped token happens to be relevant.

Plan:
- **§1 Baseline** — reproduce the 12.2 head-vs-residual table from the *stored* (v1) verdicts.
- **§2 Re-grade** — for each source (residual / goal-head) × mode (faithfulness / causality),
  take `N=16` stored-`YES` read-outs **per pivot position** and re-grade with the v2 oracle; log
  how many survive as `YES` and where the rest land (`HEAVY_LOOP` / `NO` / `NO_INFO` / `MALFORMED`).

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys, json, re, random
from pathlib import Path
from collections import Counter, defaultdict

sys.path.append("../..")

import numpy as np
import pandas as pd
from rich.console import Console
from rich.table import Table
from rich.box import SIMPLE_HEAVY

from scripts.evaluation import faithfulness_io as fio
from src.utils import env_utils
from src.utils.oracle_llms import ask_claude

console = Console(highlight=False)
def rprint(*a, **k): console.print(*a, **k)

RES   = env_utils.DEFAULT_RESULTS_DIR
ROOT  = fio.goalscope_root(RES)
MODEL = "Qwen2.5-Coder-32B-Instruct"
ORACLE_MODEL = "claude-sonnet-4-6"          # same judge as eval_goalscope
ANCHORS = ["prev_end", "indent", "var", "eq", "first_impl"]

# The two read-out SOURCES compared throughout, at the shared setup (α=0.1, n=512, seed=123).
LEAF = ("layers=35-46__nsteer=all__maxnew=64__span=rule__dtype=int__nspec=512__nc=3-4-5__seed=123")
SOURCE_LEAF = {
    "residual":  ("resid_norm",     "0.1", f"heads=resid:L25-56:mean__{LEAF}"),
    "goal-head": ("additive_norm",  "0.1", f"heads=goal__{LEAF}"),
}

def run_dir(mode: str, source: str) -> Path:
    strat, alpha, leaf = SOURCE_LEAF[source]
    return ROOT / MODEL / mode / strat / alpha / leaf

def load_records(mode: str, source: str) -> pd.DataFrame:
    """Full records for one (mode, source) leaf — keeps readout/impl_span for re-grading."""
    p = run_dir(mode, source) / "records.jsonl"
    rows = [json.loads(l) for l in open(p) if l.strip()]
    return pd.DataFrame(rows)

# sanity: every leaf is on disk
for mode in ("faithfulness", "causality"):
    for source in SOURCE_LEAF:
        d = run_dir(mode, source)
        assert (d / "records.jsonl").exists(), f"missing: {d}"
print("all 4 leaves present:", MODEL)


all 4 leaves present: Qwen2.5-Coder-32B-Instruct


## §1 Baseline — head encoding vs full residual (v1 stored verdicts)

Reproduces the 12.2 §1 table: `hard_yes` (overall + per pivot position) for the goal-head encoding
vs the full-residual patchscope, on the matched spec subset, using the verdicts already on disk
(v1 lenient oracle). This is the *"before"* we want to tighten.

In [2]:
def hard_yes_table(mode: str) -> pd.DataFrame:
    """overall + per-anchor YES-rate for both sources on their matched spec subset."""
    dfs = {s: load_records(mode, s) for s in SOURCE_LEAF}
    common = set.intersection(*[set(df.spec.unique()) for df in dfs.values()])
    rows = {}
    for s, df in dfs.items():
        d = df[df.spec.isin(common)].copy()
        d["yes"] = (d.hard == "YES").astype(float)
        by = d.groupby("anchor").yes.mean()
        rows[s] = {"overall": d.yes.mean(), **{a: by.get(a, np.nan) for a in ANCHORS}}
    print(f"[{mode}] matched specs: {len(common)}")
    return pd.DataFrame(rows).T[["overall", *ANCHORS]]

def show_df(df, title):
    t = Table(title=title, box=SIMPLE_HEAVY)
    t.add_column("", style="bold")
    for c in df.columns: t.add_column(c, justify="right")
    for lbl, r in df.iterrows():
        t.add_row(str(lbl), *["—" if (isinstance(v, float) and np.isnan(v)) else f"{v:.3f}"
                              for v in r])
    rprint(t)

for mode in ("faithfulness", "causality"):
    show_df(hard_yes_table(mode), f"§1 v1 hard_yes — {mode} · {MODEL} · α=0.1 · n=512")


[faithfulness] matched specs: 512


  §1 v1 hard_yes — faithfulness · Qwen2.5-Coder-32B-Instruct · α=0.1 ·  
                                 n=512                                  
                                                                        
              overall   prev_end   indent     var      eq   first_impl  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  residual      0.240      0.003    0.058   0.054   0.613        0.471  
  goal-head     0.674      0.331    0.577   0.491   0.986        0.986

[causality] matched specs: 512


§1 v1 hard_yes — causality · Qwen2.5-Coder-32B-Instruct · α=0.1 · n=512 
                                                                        
              overall   prev_end   indent     var      eq   first_impl  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  residual      0.513      0.226    0.367   0.414   0.739        0.817  
  goal-head     0.800      0.586    0.838   0.687   0.912        0.979

## §2 Oracle prompts — v2 (adds `HEAVY_LOOP`)

Copied from `eval_goalscope.py` and edited:
- **new `HEAVY_LOOP` verdict**, evaluated *before* `YES`, for a read-out dominated by a repeating
  cycle of ≲5 words — even if the looped token is relevant.
- dropped the v1 carve-out *"looping text is still YES as long as the repeated content is relevant."*
- `YES` now explicitly requires an intelligible descriptive fragment, not a stuck loop.

Everything else (the noisy-single-token framing, the wrong-specifics / partial-coverage leniency)
is unchanged, so a non-looping read-out is judged exactly as before.

In [ ]:
FAITH_HARD_SYS_V2 = r"""You are grading a lightweight interpretability probe (a patchscope). The DESCRIPTION below was read out of a model's hidden state at a SINGLE token while the model was writing one line of Python CODE that checks some property of its input. Because it is squeezed out of one token under a steering/patching setup, the description is usually NOISY: it may be wrapped in chat scaffolding (like 'Sure, but first let us ...'), contain code fences or half-written code, be cut off mid-sentence, or repeat and loop. Look PAST all of that surface noise and grade only the SUBSTANCE.

The question: does the description surface ANYTHING genuinely relevant to what the CODE is doing — any relevant operation, quantity, sub-step, or property of the input? Pulling even a small correct fragment out of a single token is a SUCCESS, so grade as leniently as possible — EXCEPT that a read-out which has collapsed into a degenerate repeating loop is NOT a success, no matter what tokens the loop contains (see HEAVY_LOOP).

Decide in this order — check MALFORMED and HEAVY_LOOP FIRST, and only then YES / NO / NO_INFO:

  MALFORMED  - the description is contentless: empty, pure punctuation or garbage, or repetition that carries NO relevant meaning.
  HEAVY_LOOP - the description is DOMINATED by a short repeating cycle: a unit of roughly five words/tokens or fewer that repeats over and over (e.g. 'value >= value >= value >= ...', 'len len len len', 'sum sum ( sum % sum == 0' on repeat, 'x % x == x % 0' on repeat). Once you collapse the repetition, little or no coherent description is left. Assign HEAVY_LOOP EVEN IF the looped fragment contains a relevant operator, name, or token — a stuck loop is not a faithful read-out. This takes PRECEDENCE over YES.
  YES        - the description contains SOME intelligible content genuinely relevant to what the code does, however partial, imprecise, or garbled — and it is NOT merely a short repeating loop.
  NO         - the description is coherent and about a specific thing, but that thing is COMPLETELY UNRELATED to what the code does. No conceptual sub-parts have been identified correctly.
  NO_INFO    - the description is coherent but too generic to indicate any specific aspect of the code (e.g. just 'doing a computation on the number').

The following do NOT count against the description (still YES, provided it is not a HEAVY_LOOP):
  - Wrong specifics: a wrong constant, threshold, bound, modulus, operator, or comparison. If the code computes some quantity and compares it to a value, naming the quantity alone is enough and the exact value can be wrong or missing. Example: the code implements `average temperature > 30`; a read-out about `computing the average temperature` is YES even though the `> 30` part is wrong or absent.
  - Wrong implementation nuance: naming a different mechanism for the same idea. Example: the read-out says `sorting with bubble sort` while the code calls .sort() or uses quicksort -> still YES.
  - Partial coverage: naming just one relevant ingredient of a multi-part check counts.

Grade on meaning, not on shared keywords. Output only YES, NO, NO_INFO, MALFORMED, or HEAVY_LOOP.
Put your final answer inside <verdict>{ans}</verdict> tags, with no other text or explanation, to facilitate automated parsing."""

CAUSAL_HARD_SYS_V2 = r"""You are grading a lightweight interpretability probe (a patchscope). The DESCRIPTION below was read out of a model's hidden state at a SINGLE token while the model was writing one line of Python CODE that checks some property of its input. The read-out method is a steering setup on a informationless C++ function stub. Because it is squeezed out of one token, the description is usually NOISY: it may be half-written code, be cut off mid-generation, be syntactically incorrect, or repeat and loop. Look PAST all of that surface noise and grade only the SUBSTANCE.

The question: does the C++ completion surface **ANYTHING GENUINELY RELEVANT** to what the CODE is doing — any relevant operation, quantity, sub-step, or property of the input? Pulling even a small correct fragment should be considered a SUCCESS, so grade leniently — EXCEPT that a completion which has collapsed into a degenerate repeating loop is NOT a success, no matter what tokens the loop contains (see HEAVY_LOOP).

Decide in this order — check MALFORMED and HEAVY_LOOP FIRST, and only then YES / NO / NO_INFO:

  MALFORMED  - the completion is contentless: empty, pure punctuation or garbage, or repetition that carries NO relevant meaning.
  HEAVY_LOOP - the completion is DOMINATED by a short repeating cycle: a unit of roughly five words/tokens or fewer that repeats over and over (e.g. 'value >= value >= value >= ...', '(value(value(value...', 'sum sum ( sum % sum == 0' on repeat, 'value == sum sum sum value ==' on repeat). Once you collapse the repetition, little or no coherent content is left. Assign HEAVY_LOOP EVEN IF the looped fragment contains a relevant operator, name, or token — a stuck loop is not a successful transfer. This takes PRECEDENCE over YES.
  YES        - the completion contains SOME intelligible content genuinely relevant to what the code does, however partial, imprecise, or garbled — and it is NOT merely a short repeating loop.
  NO         - the completion is coherent and about a specific thing, but that thing is COMPLETELY UNRELATED to what the code does. No conceptual sub-parts have been identified correctly.
  NO_INFO    - the completion is coherent but too generic to indicate any specific aspect of the code (e.g. just 'doing a computation on the number').

The following do NOT count against the completion (still YES, provided it is not a HEAVY_LOOP):
  - Wrong specifics: a wrong constant, threshold, bound, modulus, operator, or comparison. If the code computes some quantity and compares it to a value, naming the quantity alone is enough and the exact value can be wrong or missing. Example: the code implements `count(word, "b") > 5`; a c++ completion that counts the number of a `substr` in the `word` should be judged YES, even if the substr != b and/or the threshold is wrong.
  - Partial coverage: naming just one relevant ingredient of a multi-part check counts. For `count(word, "b") > 5`, a c++ completion that only counts the number of `substr` in the `word` is still YES, even if it doesn't check the threshold.
  - Different implementation nuance: different implementation strategies for the same idea. Example: the python code implements nth fibonacci recursively; a c++ completion implements it iteratively. Still YES, because the underlying property is the same.

Finally, grade on meaning, not on shared keywords. Output only YES, NO, NO_INFO, MALFORMED, or HEAVY_LOOP.
Put your final answer inside <verdict>{ans}</verdict> tags, with no other text or explanation, to facilitate automated parsing."""


VERDICTS = ("YES", "NO", "NO_INFO", "MALFORMED", "HEAVY_LOOP")

def parse_verdict_v2(resp: str) -> str:
    """Prefer <verdict>TOKEN</verdict>; fall back to the last bare verdict token. Adds HEAVY_LOOP."""
    m = re.search(r"<verdict>\s*(HEAVY_LOOP|MALFORMED|NO_INFO|YES|NO)\s*</verdict>", resp, re.I)
    if m:
        return m.group(1).upper()
    hits = re.findall(r"\b(HEAVY_LOOP|MALFORMED|NO_INFO|YES|NO)\b", resp.upper())
    return hits[-1] if hits else "UNPARSED"

def hard_eval_v2(readout: str, impl_span: str, mode: str) -> str:
    """Re-grade one read-out with the v2 rubric. Same user-prompt shape as eval_goalscope."""
    if not readout or not readout.strip():
        return "MALFORMED"
    sys_prompt = CAUSAL_HARD_SYS_V2 if mode == "causality" else FAITH_HARD_SYS_V2
    if mode == "causality":
        prompt = f"Python (gold): {impl_span}\nC++ (read-out): {readout.strip()}\nAnswer:"
    else:
        prompt = f"Code: {impl_span}\nDescription: {readout.strip()}\nAnswer:"
    resp = ask_claude(prompt=prompt, system_prompt=sys_prompt, max_tokens=512,
                      use_cache=True, model_name=ORACLE_MODEL)
    return parse_verdict_v2(resp)

print("v2 prompts + eval ready. verdicts:", VERDICTS)


v2 prompts + eval ready. verdicts: ('YES', 'NO', 'NO_INFO', 'MALFORMED', 'HEAVY_LOOP')


## §3 Re-grade `N=16` stored-`YES` read-outs per pivot position

For every (source × mode × pivot position), sample `N` read-outs that v1 scored `YES` (seeded), and
re-grade with the v2 oracle. **Survival** = still `YES`; the rest are demoted (mostly to
`HEAVY_LOOP` for `resid_norm`). This isolates how much of the v1 residual YES-rate was degenerate
loops.

In [4]:
N_PER_ANCHOR = 32
SAMPLE_SEED  = 123

def sample_yes(mode: str, source: str, n: int, seed: int) -> pd.DataFrame:
    """Up to `n` stored-YES records per anchor, deterministically sampled."""
    df = load_records(mode, source)
    yes = df[df.hard == "YES"]
    picks = []
    for a in ANCHORS:
        g = yes[yes.anchor == a]
        take = g.sample(min(n, len(g)), random_state=seed) if len(g) else g
        picks.append(take)
    out = pd.concat(picks, ignore_index=True) if picks else yes.iloc[:0]
    return out[["spec", "anchor", "readout", "impl_span", "condition"]]

def regrade(mode: str, source: str, bar, n=N_PER_ANCHOR, seed=SAMPLE_SEED) -> pd.DataFrame:
    s = sample_yes(mode, source, n, seed).copy()
    bar.set_postfix_str(f"{mode[:5]}/{source}")
    verdicts = []
    for r in s.itertuples():
        verdicts.append(hard_eval_v2(r.readout, r.impl_span, mode))
        bar.update(1)
    s["v2"], s["mode"], s["source"] = verdicts, mode, source
    return s

# ~4 x 5 x 16 = 320 cached oracle calls; re-runs hit the cache.
from tqdm.auto import tqdm
_combos = [(m, src) for m in ("faithfulness", "causality") for src in SOURCE_LEAF]
_total = sum(len(sample_yes(m, src, N_PER_ANCHOR, SAMPLE_SEED)) for m, src in _combos)
with tqdm(total=_total, desc="re-grading (v2)") as bar:
    graded = pd.concat([regrade(m, src, bar) for m, src in _combos], ignore_index=True)
print("re-graded rows:", len(graded))
graded.groupby(["mode", "source", "v2"]).size().unstack("v2").fillna(0).astype(int)

re-grading (v2):   0%|          | 0/615 [00:00<?, ?it/s]

re-graded rows: 615


v2                      HEAVY_LOOP  MALFORMED  NO  NO_INFO  YES
mode         source                                            
causality    goal-head           0          0  12        2  146
             residual          109          3   3        5   40
faithfulness goal-head           1          0  16        1  142
             residual          120          2   2        3    8

In [5]:
# ── survival table: of the v1-YES sample, how many stay YES under v2, per pivot position ──
def survival_table(mode: str) -> pd.DataFrame:
    g = graded[graded["mode"] == mode]
    rows = {}
    for source in SOURCE_LEAF:
        gs = g[g.source == source]
        rec = {}
        for a in ANCHORS:
            ga = gs[gs.anchor == a]
            rec[a] = np.nan if len(ga) == 0 else (ga.v2 == "YES").mean()
        rec["overall"] = np.nan if len(gs) == 0 else (gs.v2 == "YES").mean()
        rows[source] = {"overall": rec["overall"], **{a: rec[a] for a in ANCHORS}}
    return pd.DataFrame(rows).T[["overall", *ANCHORS]]

for mode in ("faithfulness", "causality"):
    show_df(survival_table(mode), f"§3 v2 YES-survival rate — {mode} (of v1-YES, N={N_PER_ANCHOR}/anchor)")

# verdict breakdown of the demoted (no-longer-YES) read-outs
print("\nwhere the v1-YES read-outs land under v2:")
graded.groupby(["mode", "source", "v2"]).size().unstack("v2").fillna(0).astype(int)


    §3 v2 YES-survival rate — faithfulness (of v1-YES, N=32/anchor)     
                                                                        
              overall   prev_end   indent     var      eq   first_impl  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  residual      0.059      0.429    0.094   0.000   0.062        0.000  
  goal-head     0.887      0.844    0.781   0.906   0.906        1.000

      §3 v2 YES-survival rate — causality (of v1-YES, N=32/anchor)      
                                                                        
              overall   prev_end   indent     var      eq   first_impl  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  residual      0.250      0.688    0.344   0.062   0.062        0.094  
  goal-head     0.912      0.938    0.906   0.844   0.969        0.906


where the v1-YES read-outs land under v2:


v2                      HEAVY_LOOP  MALFORMED  NO  NO_INFO  YES
mode         source                                            
causality    goal-head           0          0  12        2  146
             residual          109          3   3        5   40
faithfulness goal-head           1          0  16        1  142
             residual          120          2   2        3    8

### Qualitative — v1-`YES` read-outs now demoted

Eyeball a handful of the demotions (mostly `→ HEAVY_LOOP`) to check the v2 rubric is firing on the
degenerate loops and not on genuine partial read-outs.

In [6]:
def show_demotions(mode: str, source: str, verdict="HEAVY_LOOP", k=8, width=170):
    g = graded[(graded["mode"] == mode) & (graded.source == source) & (graded.v2 == verdict)]
    t = Table(title=f"{mode} · {source} · v1=YES → v2={verdict}  ({len(g)} total)",
              box=SIMPLE_HEAVY, show_lines=True)
    t.add_column("anchor", style="bold"); t.add_column("impl_span"); t.add_column("readout")
    for r in g.head(k).itertuples():
        t.add_column
        t.add_row(r.anchor, r.impl_span[:60], r.readout[:width].replace("\n", " "))
    rprint(t)

show_demotions("causality", "residual", "HEAVY_LOOP")
show_demotions("faithfulness", "residual", "HEAVY_LOOP")


show_demotions("causality", "goal-head", "NO")
show_demotions("faithfulness", "goal-head", "NO")

                            causality · residual · v1=YES → v2=HEAVY_LOOP  (109 total)                             
                                                                                                                   
  anchor     impl_span                                           readout                                           
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  prev_end   c = int(n**0.5)**2 == n                             bool check(auto value){    return value > 10      
                                                                 return c = value %    d = c == e    return d      
                                                                 return c     return d     return e     return f   
                                                                 return g     return h                             
                                                                                                                   
  prev_end   b = n >= 50                                         bool check(auto value){    return   value>        
                                                                 return   value<   return   value>   return        
                                                                 value<    return   value>    return   value<      
                                                                 return   value>    return   valu                  
                                                                                                                   
  indent     b = sum(i for i in range(1, n) if n % i == 0) > n   bool check(auto value){ return value >  return    
                                                                 sum =  sum = 0 for i in sum = 0 for i in sum = 0  
                                                                 for i in sum = 0 for i in sum = 0 for i in sum =  
                                                                 0 for i in sum = 0 for i                          
                                                                                                                   
  indent     a = n >= 1                                          bool check(auto value){ return value >  return    
                                                                 value > 5; return value > 5; return value > 5;    
                                                                 return value > 5; return value > 5; return value  
                                                                 > 5; return value > 5; retu                       
                                                                                                                   
  indent     b = gcd(n, 10) == 1                                 bool check(auto value){ return value > 0; return  
                                                                 return value % 2 == 0 return return value % 3     
                                                                 return return return value % 4 return return      
                                                                 return return value % 5 return                    
                                                                                                                   
  indent     c = n <= 500                                        bool check(auto value){ return value >  return    
                                                                 value < 5; return result = (value > 5 && value <  
                                                                 10; return result; return result = (value < 5;    
                                                                 return result = (value >  r                       
                                                                                                                   
  indent     c = n >= 1                                 

                           faithfulness · residual · v1=YES → v2=HEAVY_LOOP  (120 total)                           
                                                                                                                   
  anchor     impl_span                                           readout                                           
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  prev_end   b = n > 0                                           I'm checking the condition ifx b c d e f g h i j  
                                                                 k l m n o p q r s t u v w x y z b c d e f g h i   
                                                                 j k l m n o p q r s t u v w x y z b c d e f g h   
                                                                                                                   
  indent     a = n >= 50                                          I line of code is checking if a condition is     
                                                                 true if x = x >  x = x > x > x = x > x = x > x =  
                                                                 x > x = x > x = x > x = x > x = x > x = x > x =   
                                                                 x > x = x > x                                     
                                                                                                                   
  indent     d = any(i * (i + 1) // 2 == n for i in range(1,      I am checking if a condition is met, typically   
             n))                                                 for some variable x, y = (x == y)  return y       
                                                                 return z = (z = (x == y)  return z  return z =    
                                                                 (z = (x == y)  return z = (z =                    
                                                                                                                   
  indent     c = n <= 200                                         I am checking if the condition is met for a      
                                                                 specific action.  return result = ( condition =   
                                                                 ( x > y  result = x < y  return result = x < y    
                                                                 return result = x < y  return                     
                                                                                                                   
  indent     b = sum(i for i in range(1, n) if n % i == 0) > n    I current_line = sum total = sum total = sum     
                                                                 total = sum total = sum total = sum total = sum   
                                                                 total = sum total = sum total = sum total = sum   
                                                                 total = sum total = sum tota                      
                                                                                                                   
  indent     a = str(n) == str(n)[::-1]                           I line of code is checking if a condition is     
                                                                 true if x = x == x return x return x == x return  
                                                                 x  x = x == x return x  return x == x return x    
                                                                 x = x == x return x  return                       
                                                                                                                   
  indent     b = n >= 0                                           I am processing the line of code that checks if  
                                                        

                                causality · goal-head · v1=YES → v2=NO  (12 total)                                 
                                                                                                                   
  anchor     impl_span                                           readout                                           
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  prev_end   c = int(n**0.5)**2 == n                             bool check(auto value){ return value > 0 &&       
                                                                 value * value == value && value % 3 == 0; }       
                                                                                                                   
  prev_end   d = eval('*'.join(str(n))) > 10                     bool check(auto value){ return value > 0 &&       
                                                                 (value % 2 == 0) && ([&]{ int sum = 1; for(int i  
                                                                 = 2; i <= value / 2; ++i) if(value % i == 0) sum  
                                                                 *= i; return sum > 1; }()) &                      
                                                                                                                   
  indent     c = n % 5 == 0                                      bool check(auto value){ return value > 0 &&       
                                                                 value % 2 == 0 && value % 3 == 0; }               
                                                                                                                   
  indent     a = n >= 10                                         bool check(auto value){ return (value > 0) &&     
                                                                 (value % 2 == 0) && (value < 10); }               
                                                                                                                   
  indent     a = n >= 0                                          bool check(auto value){ return (value > 0) &&     
                                                                 (value % 2 == 0) && (value / 10 > 0); }           
                                                                                                                   
  var        b = n >= 1000                                       bool check(auto value){ return value > 10 &&      
                                                                 isdigit(value); }                                 
                                                                                                                   
  var        b = any(n == i * (i + 1) // 2 for i in range(1,     bool check(auto value){ return (value > 0) &&     
             n))                                                 ((int)sqrt(value) * (int)sqrt(value) == value)    
                                                                 && ((value % 2) == 0); }                          
                                                                                                                   
  var        a = all(gcd(n, d) == 1 for d in [3, 4])             bool check(auto value){ return (value % 2 == 0)   
                                                                 && (value > 1) && (value != 3); }

                               faithfulness · goal-head · v1=YES → v2=NO  (16 total)                               
                                                                                                                   
  anchor     impl_span                                           readout                                           
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  prev_end   b = n > 0                                           I'm checking if the variable `a` is true.         
                                                                                                                   
  prev_end   d = sum(1 for i in range(1, n + 1) if n % i == 0)   I'm checking if the current number is greater     
             > 4                                                 than 1 and all numbers up to its square root      
                                                                 divide it without a remainder, indicating it's    
                                                                 prime.                                            
                                                                                                                   
  prev_end   c = sum(int(digit) for digit in str(n)) % 3 == 0    I'm checking if the sum of the digits of the      
                                                                 number is greater than the original number.       
                                                                                                                   
  prev_end   b = n >= 0                                          Sure, but I need to see the specific line of      
                                                                 code and have context for each condition (e.g.,   
                                                                 whether it's Python). Here's a quick breakdown:   
                                                                 1. `import sys; is_valid = s                      
                                                                                                                   
  indent     a = n <= 50                                         Sure, please provide the line of code for me to   
                                                                 evaluate.  a = x < 5 b = a and x > 0 c = b and x  
                                                                 % 2 == 0  Here are the evaluations: - `a` checks  
                                                                 if `x` is less than 5. -                          
                                                                                                                   
  indent     e = int(n**0.5)**2 == n                             I'm checking if the variable `x` is equal to the  
                                                                 square root of 10, and storing the result in      
                                                                 `y`.                                              
                                                                                                                   
  indent     a = bin(n).count('1') > 2                           I'm checking if the current bit of the variable   
                                                                 `x` is set to 1 by using bitwise operations and   
                                                                 logical checks.                                   
                                                                                                                   
  indent     b = sum(int(digit) for digit in str(n)) > 10        I'm checking if the sum of the digits of the      
                                                                 number is equal to the product of its digits.

In [14]:
impl_span = "c = sum(int(digit) for digit in str(n)) % 3 == 0"
readout = "I'm checking if the sum of the digits of the number is greater than the original number."

resp = ask_claude(
    prompt=f"Code: {impl_span}\nDescription: {readout.strip()}\nAnswer:",
    system_prompt=FAITH_HARD_SYS_V2,
    max_tokens=512,
    use_cache=False,
    model_name=ORACLE_MODEL,
)
resp

'<verdict>NO</verdict>'

### Knobs to refine
- `N_PER_ANCHOR`, `SAMPLE_SEED` — sample size / draw per pivot position.
- `FAITH_HARD_SYS_V2` / `CAUSAL_HARD_SYS_V2` — the `HEAVY_LOOP` threshold (currently *"≲5-word
  cycle, dominant"*). Loosen/tighten the cycle length or the *"dominated by"* wording here.
- To also re-grade the **non-YES** buckets (does v2 pull anything the *other* way?), widen
  `sample_yes` to all verdicts.
- Once the rubric is settled: port `HEAVY_LOOP` + the reordered decision list back into
  `scripts/evaluation/eval_goalscope.py` (`FAITH_HARD_SYS`, `CAUSAL_HARD_SYS`, `_parse_verdict`,
  and `compute_summary` if `HEAVY_LOOP` should be tracked separately), then re-score the runs.